In [1]:
import os
import copy
import pickle
import sympy
import functools
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from error_injection import MissingValueError, SamplingError, Injector
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.metrics import mutual_info_score, auc, roc_curve, roc_auc_score, f1_score
from scipy.optimize import minimize as scipy_min
from scipy.spatial import ConvexHull
from scipy.optimize import minimize, Bounds, linprog
from sympy import Symbol as sb
from sympy import lambdify
from tqdm.notebook import trange,tqdm
from IPython.display import display,clear_output
from random import choice
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import _tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score

class style():
    RED = '\033[31m'
    GREEN = '\033[32m'
    BLUE = '\033[34m'
    RESET = '\033[0m'

np.random.seed(1)

# ignore all the warnings
import warnings
warnings.filterwarnings('ignore')

from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
forest_fires = fetch_ucirepo(id=162) 

In [2]:
def load_ins_cleaned():
    # fetch dataset
    auto_mpg = pd.read_csv('datasets/insurance.csv').drop('sex', axis=1).drop('smoker', axis=1).drop('region', axis=1).replace('?', np.nan)
    features = ['age', 'bmi', 'children']
    X = auto_mpg[features].astype(float)
    y = auto_mpg['charges']
    
    # assumed gt imputation
    imputer = KNNImputer(n_neighbors=10)
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
    X_train = copy.deepcopy(X_train).reset_index(drop=True)
    X_test = copy.deepcopy(X_test).reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)

    return X_train, X_test, y_train, y_test

X_train_ins, X_test_ins, y_train_ins, y_test_ins = load_ins_cleaned()
len(X_train_ins)

1070

In [3]:
# first impute the data and make it hypothetically clean
def load_mpg_cleaned():
    # fetch dataset
    auto_mpg = pd.read_csv('datasets/auto-mpg.csv').drop('car name', axis=1).replace('?', np.nan)
    
    features = ['cylinders', 'displacement', 'horsepower', 'weight',
                'acceleration', 'model year', 'origin']
    X = auto_mpg[features].astype(float)
    y = auto_mpg['mpg']
    
    # assumed gt imputation
    imputer = KNNImputer(n_neighbors=10)
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
    X_train = copy.deepcopy(X_train).reset_index(drop=True)
    X_test = copy.deepcopy(X_test).reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)

    return X_train, X_test, y_train, y_test
X_train_mpg, X_test_mpg, y_train_mpg, y_test_mpg = load_mpg_cleaned()
len(X_train_mpg)

318

In [4]:
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
boston_df = pd.read_csv('datasets/housing.csv', header=None, delimiter=r"\s+", names=column_names)
boston_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     506 non-null    float64
 1   ZN       506 non-null    float64
 2   INDUS    506 non-null    float64
 3   CHAS     506 non-null    int64  
 4   NOX      506 non-null    float64
 5   RM       506 non-null    float64
 6   AGE      506 non-null    float64
 7   DIS      506 non-null    float64
 8   RAD      506 non-null    int64  
 9   TAX      506 non-null    float64
 10  PTRATIO  506 non-null    float64
 11  B        506 non-null    float64
 12  LSTAT    506 non-null    float64
 13  MEDV     506 non-null    float64
dtypes: float64(12), int64(2)
memory usage: 55.5 KB


In [5]:
#Useful functions
symbol_id = -1
def create_symbol(suffix=''):
    global symbol_id
    symbol_id += 1
    name = f'e{symbol_id}_{suffix}' if suffix else f'e{symbol_id}'
    return sympy.Symbol(name=name)


#scaler_symbols = set([sb(f'k{i}') for i in range(X_train.shape[1]+1)])
#linearization_dict = dict()
#reverse_linearization_dict = dict()

def inject_sensitive_ranges(X, y, uncertain_attr, uncertain_num, boundary_indices, uncertain_radius_pct=None, 
                  uncertain_radius=None, seed=42):
    global symbol_id
    symbol_id = -1
    
    X_extended = np.append(np.ones((len(X), 1)), X, axis=1)
    ss = StandardScaler()
    X_extended[:, 1:] = ss.fit_transform(X_extended[:, 1:])
    X_extended_symb = sympy.Matrix(X_extended)
    
    if not(uncertain_attr=='y'):
        uncertain_attr_idx = X.columns.to_list().index(uncertain_attr) + 1
        if not(uncertain_radius):
            uncertain_radius = uncertain_radius_pct*(np.max(X_extended[:, uncertain_attr_idx])-\
                                                     np.min(X_extended[:, uncertain_attr_idx]))
    else:
        if not(uncertain_radius):
            uncertain_radius = uncertain_radius_pct*(y_train.max()-y_train.min())[0]
    
    np.random.seed(seed)
    uncertain_indices = boundary_indices[:uncertain_num]
    y_symb = sympy.Matrix(y)
    symbols_in_data = set()
    #print(uncertain_indices)
    for uncertain_idx in uncertain_indices:
        new_symb = create_symbol()
        symbols_in_data.add(new_symb)
        if uncertain_attr=='y':
            y_symb[uncertain_idx] = y_symb[uncertain_idx] + uncertain_radius*new_symb
        else:
            X_extended_symb[uncertain_idx, uncertain_attr_idx] = X_extended_symb[uncertain_idx, uncertain_attr_idx] + uncertain_radius*new_symb
    return X_extended_symb, y_symb, symbols_in_data, ss

# if interval=True, use interval arithmetic, otherwise use zonotopes
def compute_robustness_ratio_sensitive_label_error(X_train, y_train, X_test, y_test, robustness_radius,
                                         uncertain_num, boundary_indices, uncertain_radius=None, 
                                         lr=0.1, seed=42, interval=True):
    X, y, symbols_in_data, ss = inject_sensitive_ranges(X=X_train, y=y_train, uncertain_attr='y', 
                                              uncertain_num=uncertain_num, boundary_indices=boundary_indices, 
                                              uncertain_radius=uncertain_radius, 
                                              uncertain_radius_pct=None, seed=seed)
    
    assert len(X.free_symbols)==0
    # closed-form
    param = (X.T*X).inv()*X.T*y
    
    if interval:
        # make param intervals
        for d in range(len(param)):
            expr = param[d]
            if not(expr.free_symbols):
                continue
            else:
                constant_part = 0
                interval_radius = 0
                for arg in expr.args:
                    if arg.free_symbols:
                        interval_radius += abs(arg.args[0])
                    else:
                        assert constant_part == 0
                        constant_part = arg
                param[d] = constant_part + create_symbol()*interval_radius
    
    test_preds = sympy.Matrix(np.append(np.ones((len(X_test), 1)), ss.transform(X_test), axis=1))*param
    robustness_ls = []
    for pred in test_preds:
        pred_range_radius = 0
        for arg in pred.args:
            if arg.free_symbols:
                pred_range_radius += abs(arg.args[0])
        if pred_range_radius <= robustness_radius:
            robustness_ls.append(1)
        else:
            robustness_ls.append(0)
    
#     print(param)
    return np.mean(robustness_ls)

In [6]:
X_train_transformed = X_train_ins.copy()
X_test_transformed = X_test_ins.copy()

columns_to_bin = ['bmi']
n_bins = {'bmi': int(np.sqrt(496))}
column_bins = {}

for col in columns_to_bin:
    # Apply binning (equal-width bins in this example)
    X_train_transformed[col], bins = pd.cut(
        X_train_ins[col], 
        bins=n_bins[col], 
        labels=False, 
        retbins=True
    )

    X_test_transformed[col] = pd.cut(
        X_test_ins[col],
        bins=bins,      # Use the same bins from X_train
        labels=False,   # Keep consistent labels
        include_lowest=True  # Ensure the lowest bin includes its boundary
    )

    X_train_transformed[col] = X_train_transformed[col].fillna(-1).astype(int)
    X_test_transformed[col] = X_test_transformed[col].fillna(-1).astype(int)

    column_bins[col] = bins

def get_complex_candidate_target_indices(df, candidate):
    columns, values, conditions = candidate
    mask = pd.Series(True, index=df.index)  # Start with a mask that selects all rows

    for col, val, cond in zip(columns, values, conditions):
        if cond == "<":
            mask &= df.iloc[:, col] < val
        elif cond == "=":
            mask &= df.iloc[:, col] == val
        elif cond == ">":
            mask &= df.iloc[:, col] > val
        else:
            raise ValueError(f"Unsupported condition: {cond}")

    return df[mask].index.tolist()


pattern_ins = ((0, 1, 2), (51.0, 0, 0.0), ('>', '>', '>'))
target_indices_of_pattern_ins = get_complex_candidate_target_indices(X_train_transformed, pattern_ins)
boundary_indices_lst = [target_indices_of_pattern_ins]

In [7]:
X_train_transformed = X_train_mpg.copy()
X_test_transformed = X_test_mpg.copy()

columns_to_bin = ['displacement', 'horsepower', 'weight', 'acceleration']
n_bins = {'displacement': int(np.sqrt(72)), 'horsepower': int(np.sqrt(87)), 'weight': int(np.sqrt(288)), 'acceleration': int(np.sqrt(86))}
column_bins = {}

for col in columns_to_bin:
    # Apply binning (equal-width bins in this example)
    X_train_transformed[col], bins = pd.cut(
        X_train_mpg[col], 
        bins=n_bins[col], 
        labels=False, 
        retbins=True
    )

    X_test_transformed[col] = pd.cut(
        X_test_mpg[col],
        bins=bins,      # Use the same bins from X_train
        labels=False,   # Keep consistent labels
        include_lowest=True  # Ensure the lowest bin includes its boundary
    )

    X_train_transformed[col] = X_train_transformed[col].fillna(-1).astype(int)
    X_test_transformed[col] = X_test_transformed[col].fillna(-1).astype(int)

    column_bins[col] = bins

def get_complex_candidate_target_indices(df, candidate):
    columns, values, conditions = candidate
    mask = pd.Series(True, index=df.index)  # Start with a mask that selects all rows

    for col, val, cond in zip(columns, values, conditions):
        if cond == "<":
            mask &= df.iloc[:, col] < val
        elif cond == "=":
            mask &= df.iloc[:, col] == val
        elif cond == ">":
            mask &= df.iloc[:, col] > val
        else:
            raise ValueError(f"Unsupported condition: {cond}")

    return df[mask].index.tolist()

pattern_mpg =  ((3, 4, 5), (9, 3, 82.0), ('<', '<', '<'))

target_indices_of_pattern_mpg = get_complex_candidate_target_indices(X_train_transformed, pattern_mpg)
boundary_indices_lst = boundary_indices_lst + [target_indices_of_pattern_mpg]

In [8]:
X = boston_df.drop(columns=['MEDV'])
y = boston_df['MEDV']
X_train_bos, X_test_bos, y_train_bos, y_test_bos = train_test_split(X, y, test_size=0.2, random_state=1)

print("Training data shape:", X_train_bos.shape)
print("Testing data shape:", X_test_bos.shape)

Training data shape: (404, 13)
Testing data shape: (102, 13)


In [9]:
X = forest_fires.data.features 
y = forest_fires.data.targets 
  
X['area'] = y['area']
#X = X.sample(frac=0.3)
y = X['area']
X = X.drop('area', axis=1)
X = X.drop('month', axis=1)
X = X.drop('day', axis=1)
  
X_train_fire, X_test_fire, y_train_fire, y_test_fire = train_test_split(X, y, test_size=0.2, random_state=1)
X_train_fire = copy.deepcopy(X_train_fire).reset_index(drop=True)
X_test_fire = copy.deepcopy(X_test_fire).reset_index(drop=True)
y_train_fire = y_train_fire.reset_index(drop=True)
y_test_fire = y_test_fire.reset_index(drop=True)

In [10]:
y_train_bos.copy()

42     25.3
58     23.3
385     7.2
78     21.2
424    11.7
       ... 
255    20.9
72     22.8
396    12.5
235    24.0
37     21.0
Name: MEDV, Length: 404, dtype: float64

In [11]:
X_train_bos, X_test_bos, y_train_bos, y_test_bos = X_train_bos.reset_index(drop=True), X_test_bos.reset_index(drop=True), y_train_bos.reset_index(drop=True), y_test_bos.reset_index(drop=True)

X_train_transformed = X_train_bos.copy()
X_test_transformed = X_test_bos.copy()

columns_to_bin = ['CRIM', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'B', 'LSTAT']
n_bins = {'CRIM': int(np.sqrt(402)), 'INDUS': int(np.sqrt(72)), 'NOX': int(np.sqrt(76)), 'RM': int(np.sqrt(366)), 'AGE': int(np.sqrt(302)), 'DIS': int(np.sqrt(339)), 'B': int(np.sqrt(287)), 'LSTAT': int(np.sqrt(366))}
column_bins = {}

for col in columns_to_bin:
    # Apply binning (equal-width bins in this example)
    X_train_transformed[col], bins = pd.cut(
        X_train_bos[col], 
        bins=n_bins[col], 
        labels=False, 
        retbins=True
    )

    X_test_transformed[col] = pd.cut(
        X_test_bos[col],
        bins=bins,      # Use the same bins from X_train
        labels=False,   # Keep consistent labels
        include_lowest=True  # Ensure the lowest bin includes its boundary
    )

    X_train_transformed[col] = X_train_transformed[col].fillna(-1).astype(int)
    X_test_transformed[col] = X_test_transformed[col].fillna(-1).astype(int)

    column_bins[col] = bins

def get_complex_candidate_target_indices(df, candidate):
    columns, values, conditions = candidate
    mask = pd.Series(True, index=df.index)  # Start with a mask that selects all rows

    for col, val, cond in zip(columns, values, conditions):
        if cond == "<":
            mask &= df.iloc[:, col] < val
        elif cond == "=":
            mask &= df.iloc[:, col] == val
        elif cond == ">":
            mask &= df.iloc[:, col] > val
        else:
            raise ValueError(f"Unsupported condition: {cond}")

    return df[mask].index.tolist()

pattern_bos = ((7, 12), (3, 5), ('<', '<'))
target_indices_of_pattern_bos = get_complex_candidate_target_indices(X_train_transformed, pattern_bos)
boundary_indices_lst = boundary_indices_lst + [target_indices_of_pattern_bos]

In [12]:
X_train_transformed = X_train_fire.copy()
X_test_transformed = X_test_fire.copy()

columns_to_bin = ['FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH']
n_bins = {'FFMC': int(np.sqrt(103)), 'DMC': int(np.sqrt(199)), 'DC': int(np.sqrt(199)), 'ISI': int(np.sqrt(112)), 'temp': int(np.sqrt(183)), 'RH': int(np.sqrt(73))}
column_bins = {}

for col in columns_to_bin:
    # Apply binning (equal-width bins in this example)
    X_train_transformed[col], bins = pd.cut(
        X_train_fire[col], 
        bins=n_bins[col], 
        labels=False, 
        retbins=True
    )

    X_test_transformed[col] = pd.cut(
        X_test_fire[col],
        bins=bins,      # Use the same bins from X_train
        labels=False,   # Keep consistent labels
        include_lowest=True  # Ensure the lowest bin includes its boundary
    )

    X_train_transformed[col] = X_train_transformed[col].fillna(-1).astype(int)
    X_test_transformed[col] = X_test_transformed[col].fillna(-1).astype(int)

    column_bins[col] = bins

def get_complex_candidate_target_indices(df, candidate):
    columns, values, conditions = candidate
    mask = pd.Series(True, index=df.index)  # Start with a mask that selects all rows

    for col, val, cond in zip(columns, values, conditions):
        if cond == "<":
            mask &= df.iloc[:, col] < val
        elif cond == "=":
            mask &= df.iloc[:, col] == val
        elif cond == ">":
            mask &= df.iloc[:, col] > val
        else:
            raise ValueError(f"Unsupported condition: {cond}")

    return df[mask].index.tolist()

pattern_fire = ((4, 8), (0, 5.8), ('>', '>'))

target_indices_of_pattern_fire = get_complex_candidate_target_indices(X_train_transformed, pattern_fire)
boundary_indices_lst = boundary_indices_lst + [target_indices_of_pattern_fire]

In [13]:
uncertain_radius_ins = 0.25*(y_train_ins.max() - y_train_ins.min())
uncertain_radius_mpg = 0.25*(y_train_mpg.max() - y_train_mpg.min())
uncertain_radius_bos = 0.25*(y_train_bos.max() - y_train_bos.min())
uncertain_radius_fire = 0.25*(y_train_fire.max() - y_train_fire.min())
uncertain_radii = [uncertain_radius_ins, uncertain_radius_mpg, uncertain_radius_bos, uncertain_radius_fire]

uncertain_percentage = 0.1
uncertain_num_ins = int(uncertain_percentage*len(y_train_ins))
uncertain_num_mpg = int(uncertain_percentage*len(y_train_mpg))
uncertain_num_bos = int(uncertain_percentage*len(y_train_bos))
uncertain_num_fire = int(uncertain_percentage*len(y_train_fire))
uncertain_numbers = [uncertain_num_ins, uncertain_num_mpg, uncertain_num_bos, uncertain_num_fire]

dataset_sizes = [len(y_train_ins), len(y_train_mpg), len(y_train_bos), len(y_train_fire)]
dataset_names = ["Insurance", "MPG", "BOS", "FIRE"]

dataset_dct = {}
dataset_dct["Insurance"] = [X_train_ins, X_test_ins, y_train_ins, y_test_ins]
dataset_dct["MPG"] = [X_train_mpg, X_test_mpg, y_train_mpg, y_test_mpg]
dataset_dct["BOS"] = [X_train_bos, X_test_bos, y_train_bos, y_test_bos]
dataset_dct["FIRE"] = [X_train_fire, X_test_fire, y_train_fire, y_test_fire]

In [17]:
y_train_ins.min(), y_train_ins.max()

(1121.8739, 63770.42801)

In [18]:
y_train_mpg.min(), y_train_mpg.max()

(9.0, 46.6)

In [19]:
y_train_bos.min(), y_train_bos.max()

(5.0, 50.0)

In [20]:
y_train_fire.min(), y_train_fire.max()

(0.0, 1090.84)

In [16]:
np.std(y_train_ins)

12076.792882011212

In [21]:
np.std(y_train_bos)

8.987846511442205

In [22]:
np.std(y_train_mpg)

7.878333517784325

In [23]:
np.std(y_train_fire)

59.916851463223004

In [24]:
def robustness_score_normalization(uncertain_numbers, uncertain_radii, dataset_sizes, boundary_indices_lst, dataset_names, dataset_dct):
    robustness_radii_10 = [] #find robustness radius that grants radii robustness ratio of 0.10 or more 
                             #(alt. use 0.5 instead {depending on closeness, this ratio may need to be larger})

    for i in range(0, len(dataset_names)):
        uncertain_number = uncertain_numbers[i]
        uncertain_radius = uncertain_radii[i]
        boundary_indices = boundary_indices_lst[i]
        X_train, X_test, y_train, y_test = dataset_dct[dataset_names[i]]

        #print("target:")
        robustness_radius= 1
        if dataset_names[i] == "Insurance":
            radius_increment = 500
        elif dataset_names[i] == "FIRE":
            radius_increment = 1
        else:
            radius_increment = 0.01

        robustness_ratio = compute_robustness_ratio_sensitive_label_error(X_train, y_train, X_test, y_test, 
                                                                    uncertain_num=uncertain_number,
                                                                    boundary_indices=boundary_indices,
                                                                    uncertain_radius=uncertain_radius, 
                                                                    robustness_radius=robustness_radius,
                                                                    interval=False)

        #print(robustness_ratio)
        with tqdm(total=500, desc=f"Finding radius for {dataset_names[i]}", leave=False) as pbar:
            while robustness_ratio < 0.25:
                robustness_radius += radius_increment
                robustness_ratio = compute_robustness_ratio_sensitive_label_error(X_train, y_train, X_test, y_test, 
                                                                    uncertain_num=uncertain_number,
                                                                    boundary_indices=boundary_indices,
                                                                    uncertain_radius=uncertain_radius, 
                                                                    robustness_radius=robustness_radius,
                                                                    interval=False)
                pbar.update(radius_increment)
                #print(robustness_ratio)
        print(robustness_radius)
        #robustness_radii_10.append(robustness_radius)
        robustness_radii_10.append(robustness_radius/np.std(y_test))

    results = {}

    mean_radius = np.mean(robustness_radii_10)
    std_radius = np.std(robustness_radii_10)
    
    max_radii = max(robustness_radii_10)
    min_radii = min(robustness_radii_10)
    for i, dataset_name in enumerate(dataset_names):
        normalized_radius = 1 - ((robustness_radii_10[i] - mean_radius) / (std_radius + 1e-8))  # Prevent division by zero
        #normalized_size = (dataset_sizes[i]/max(dataset_sizes))
        #robustness_score = 0.5*normalized_radius + 0.5*normalized_size    
        robustness_score = normalized_radius
        #print(f"Normalized robustness score for {dataset_name} dataset is {robustness_score:.4f}")
        results[dataset_name] = robustness_score

    items = list(sorted(results.items(), key=lambda x: x[1]))
        
    for item in items:
        print(f"Normalized robustness score for {item[0]} dataset is {item[1]:.4f}")
    
    return results

result = robustness_score_normalization(uncertain_numbers, uncertain_radii, dataset_sizes, boundary_indices_lst, dataset_names, dataset_dct)

Finding radius for Insurance:   0%|          | 0/500 [00:00<?, ?it/s]

2001


Finding radius for MPG:   0%|          | 0/500 [00:00<?, ?it/s]

1.7300000000000006


Finding radius for BOS:   0%|          | 0/500 [00:00<?, ?it/s]

2.219999999999996


Finding radius for FIRE:   0%|          | 0/500 [00:00<?, ?it/s]

52
Normalized robustness score for FIRE dataset is -0.7185
Normalized robustness score for MPG dataset is 1.4541
Normalized robustness score for BOS dataset is 1.4887
Normalized robustness score for Insurance dataset is 1.7757


In [25]:
result

{'Insurance': 1.7757149634663207,
 'MPG': 1.4541167860308875,
 'BOS': 1.488669370797243,
 'FIRE': -0.7185011202944505}

In [26]:
result = robustness_score_normalization(uncertain_numbers[1:], uncertain_radii[1:], dataset_sizes[1:], boundary_indices_lst[1:], dataset_names[1:], dataset_dct)

Finding radius for MPG:   0%|          | 0/500 [00:00<?, ?it/s]

1.7300000000000006


Finding radius for BOS:   0%|          | 0/500 [00:00<?, ?it/s]

2.219999999999996


Finding radius for FIRE:   0%|          | 0/500 [00:00<?, ?it/s]

52
Normalized robustness score for FIRE dataset is -0.4141
Normalized robustness score for MPG dataset is 1.6903
Normalized robustness score for BOS dataset is 1.7238


In [27]:
result

{'MPG': 1.6903070006412877,
 'BOS': 1.7237744876922563,
 'FIRE': -0.41408148833354375}

Normalized robustness score for Insurance dataset is 0.5000

Normalized robustness score for MPG dataset is 0.6474

{'Insurance': 0.5, 'MPG': 0.6473962077641984}